# TP : Implémentation de l'Algorithme d'Estimation de Phase Quantique (QPE)

L'algorithme d'estimation de phase quantique (QPE) permet d'estimer la phase $\theta$ d'une valeur propre associée à un vecteur propre d'un opérateur unitaire $U$ tel que : 
$$U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$$

**Objectif de cette première partie :**
Vous allez construire pas à pas la classe `MonQPE`. Votre travail consiste à compléter les méthodes individuelles permettant de construire les différentes parties du circuit quantique. La méthode `run` (déjà implémentée) se chargera d'assembler votre travail, d'afficher le circuit proprement et de l'exécuter.

In [ ]:
# Exécutez cette cellule pour importer les librairies nécessaires
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT
from qiskit.primitives import Sampler
from IPython.display import display

: 

## 1. Complétez la construction du circuit

Vous devez compléter les méthodes suivantes dans la classe ci-dessous :
1. **`apply_initial_state`** : Prépare l'état propre $|\psi\rangle$ sur le registre principal (cible).
2. **`apply_hadamard`** : Applique une porte Hadamard sur tous les qubits du registre d'ancilla (pour créer la superposition).
3. **`apply_controlled_unitary`** : Applique les opérations $U^{2^j}$ contrôlées par le $j$-ème qubit d'ancilla. *Indice : Utilisez la méthode `.control()` sur votre porte unitaire et ajoutez-la au circuit avec `.append()`*.
4. **`apply_inverse_qft`** : Applique la transformée de Fourier quantique inverse. *Indice : Vous pouvez utiliser `QFT(num_qubits, inverse=True).to_instruction()`*.
5. **`measure`** : Mesure le registre d'ancilla dans le registre classique.

In [ ]:
class MonQPE:
    def __init__(self, unitary_gate, state_prep_circuit, num_ancilla):
        """
        Initialise l'algorithme QPE.
        :param unitary_gate: La porte quantique U (objet Gate de Qiskit).
        :param state_prep_circuit: Circuit préparant l'état propre |\psi>.
        :param num_ancilla: Nombre de qubits pour le registre de comptage (précision).
        """
        self.unitary = unitary_gate
        self.state_prep = state_prep_circuit
        self.num_ancilla = num_ancilla
        self.num_target = state_prep_circuit.num_qubits
        
        # Création des registres
        self.anc_register = QuantumRegister(self.num_ancilla, 'ancilla')
        self.target_register = QuantumRegister(self.num_target, 'target')
        self.classical_register = ClassicalRegister(self.num_ancilla, 'meas')
        
        # Initialisation du circuit global
        self.qc = QuantumCircuit(self.anc_register, self.target_register, self.classical_register)

    def apply_initial_state(self):
        """ ÉTAPE 1 : Applique la préparation de l'état sur le registre cible. """
        # --- À COMPLÉTER ---
        # Indice : utilisez self.qc.compose(...)
        pass
        
    def apply_hadamard(self):
        """ ÉTAPE 2 : Applique des portes H sur le registre d'ancilla. """
        # --- À COMPLÉTER ---
        pass

    def apply_controlled_unitary(self):
        """ ÉTAPE 3 : Applique les portes U contrôlées (U^{2^j}). """
        # --- À COMPLÉTER ---
        # Pour chaque qubit d'ancilla j de 0 à num_ancilla-1 :
        # 1. Créez la porte U à la puissance 2^j (ex: self.unitary.power(2**j))
        # 2. Transformez-la en porte contrôlée
        # 3. Ajoutez-la au circuit (self.qc.append(...))
        pass

    def apply_inverse_qft(self):
        """ ÉTAPE 4 : Applique la Transformée de Fourier Quantique Inverse. """
        # --- À COMPLÉTER ---
        pass

    def measure(self):
        """ ÉTAPE 5 : Mesure le registre d'ancilla. """
        # --- À COMPLÉTER ---
        pass

    def run(self, shots=1024):
        """ 
        MÉTHODE FOURNIE : Assemble le circuit, l'affiche et l'exécute.
        """
        # 1. Construction du circuit en appelant vos méthodes
        self.apply_initial_state()
        self.qc.barrier() # Barrière visuelle
        
        self.apply_hadamard()
        self.qc.barrier()
        
        self.apply_controlled_unitary()
        self.qc.barrier()
        
        self.apply_inverse_qft()
        self.qc.barrier()
        
        self.measure()

        # 2. Affichage graphique du circuit
        print("Circuit QPE généré :")
        # L'utilisation de to_instruction() ou .control() dans vos méthodes 
        # garantit que U apparaît comme un bloc unique !
        display(self.qc.draw('mpl', fold=-1))

        # 3. Exécution avec le Sampler
        print("\nExécution en cours...")
        sampler = Sampler()
        job = sampler.run([self.qc], shots=shots)
        result = job.result()
        
        # Récupération des counts (dictionnaire des états mesurés)
        counts = result.quasi_dists[0].binary_probabilities()
        
        print("Résultats bruts (Probabilités) :", counts)
        return counts

## 2. Phase de Test

Pour vérifier que votre classe fonctionne, nous allons définir un cas d'usage simple :
- La porte unitaire $U$ sera une porte de Phase quantique $P(\theta)$ avec $\theta = \pi/4$.
- En unités de $2\pi$, la phase attendue est donc $0.125$ (soit `001` en binaire pour 3 ancillas).
- L'état propre associé à cette porte est l'état $|1\rangle$ (on appliquera donc une porte $X$ pour la préparation d'état).

In [ ]:
from qiskit.circuit.library import PhaseGate

# 1. Définition de l'opérateur U (Porte de Phase)
# La valeur propre associée à |1> sera exp(i * pi/4)
# La phase est donc theta = (pi/4) / (2*pi) = 1/8 = 0.125
theta_angle = np.pi / 4
U_gate = PhaseGate(theta_angle)
# On donne un nom explicite pour l'affichage du bloc
U_gate.name = "U(pi/4)"

# 2. Préparation de l'état propre |1> (Porte X)
prep_circuit = QuantumCircuit(1)
prep_circuit.x(0)

# 3. Création de l'instance de votre classe (Précision : 3 qubits)
qpe_instance = MonQPE(unitary_gate=U_gate, state_prep_circuit=prep_circuit, num_ancilla=3)

# 4. Exécution (construit, affiche et simule)
counts = qpe_instance.run(shots=1000)

# Note pour la lecture des résultats : 
# L'état binaire mesuré le plus probable devrait être '001'.
# En binaire fractionnaire, 0.001 = 1/8 = 0.125 !